# Flow Chart Pengerjaan:
1. Cover Image
2. DCT & Quantization
3. Block Smoothness Estimation & Sorting
4. Zigzag Scan
5. NACP Construction
6. Adaptive Hexagonal Payload Assignment
7. Hexagonal Turtle Shell Embedding
8. Stego DCT Coefficients
9. Entropy Coding
10. Stego Image

In [13]:
!pip install jpeglib numpy matplotlib opencv-python-headless scipy scikit-image seaborn pandas tqdm import-ipynb


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from PIL import Image
from math import ceil, floor
from performance import psnr, fsi, ssim
from zigzag import zigzag, inverse_zigzag
import jpeglib
import numpy as np
import import_ipynb
import turtleShell
import FrequencyDomain as FD

In [15]:
def change_image_QF(image_path, target_qf):
    im = jpeglib.read_dct(image_path)
    old_qt = im.qt[0]

    # From QF 100 to target QF
    dequantized = im.Y.astype(np.float64) * old_qt
    new_coefficients = np.round(dequantized / FD.custom_q_mat(target_qf)).astype(np.int16)
    
    # Update image object
    im.Y[:] = new_coefficients
    im.qt[0] = FD.custom_q_mat(target_qf)
    
    output_path = f"{image_path}_requantized_qf{target_qf}.jpeg"
    im.write_dct(output_path)
    print(f"Image {image_path} re-quantized to QF {target_qf} and saved to {output_path}")

In [16]:
def get_nacp(sorted_coefficients):
    valid_nacp = []
    for zigzag_coeff in sorted_coefficients:
        ac_coeffs = zigzag_coeff[1:] # AC coefficients
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) >= 1)[0]
        non_zero_ac = [ac_coeffs[i] for i in non_zero_indices]

        for i in range(0, len(non_zero_ac) - 1, 2):
            x = int(non_zero_ac[i])
            y = int(non_zero_ac[i+1])
            if x != 0 and y != 0:
                valid_nacp.append((x, y))

    return valid_nacp

In [17]:
def replace_nacp(sorted_coefficients, nacp_coords):
    pair_index = 0

    for zigzag_coeff in sorted_coefficients:
        ac_coeffs = zigzag_coeff[1:]  
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) >= 1)[0]
        non_zero_ac = [ac_coeffs[i] for i in non_zero_indices]

        for idx in range(0, len(non_zero_ac) - 1, 2):
            if pair_index < len(nacp_coords):
                new_x, new_y = nacp_coords[pair_index]
                i1, i2 = non_zero_indices[idx], non_zero_indices[idx + 1]
                ac_coeffs[i1] = float(new_x)
                ac_coeffs[i2] = float(new_y)
                pair_index += 1
            else: break
        zigzag_coeff[1:] = ac_coeffs

    return sorted_coefficients

In [18]:
def get_quantized_coefficients(image_path):
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, _, _  = im.Y.shape
    sorted_coeffs = []
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block = im.Y[i, j]
            zigzag_coeffs = zigzag(block)
            sorted_coeffs.append(zigzag_coeffs)
    return sorted_coeffs

In [19]:
def construct_stego_file(image_path, new_coeffs):
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, v_block_size, h_block_size  = im.Y.shape
    idx = 0
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block_coeffs = new_coeffs[idx]
            block = inverse_zigzag(block_coeffs, v_block_size, h_block_size)
            im.Y[i, j] = block
            idx += 1
    ori_path = image_path.split(".jpeg")[0].split("/")[-1]
    stego_folder = "stego-images/"
    output_path = f"{stego_folder}stego_{ori_path}.jpeg"
    im.write_dct(output_path)
    print(f"Image with secret data is saved to {output_path}")

In [20]:
def data_hiding_process(secret_data, nacp_coord="", mode="8N"):
    secret_data = secret_data + '\0'
    data_bin = ''.join(format(ord(c), '08b') for c in secret_data)
    lendata = len(data_bin)
    print(f"Secret Data: {secret_data}")
    print(f"Panjang Bit Secret Data: {lendata}")

    decimals = []
    for i in range(0, len(data_bin), 3):
        group = data_bin[i:i+3].ljust(3, '0')
        decimals.append(int(group, 2))

    _, shells, cell_to_shells = turtleShell.init(mode) 
    if len(decimals) > len(nacp_coord):
        print("Warning: Not enough NACP coordinates to embed all data.")
        
    for i in range(len(decimals)):
        x, y = nacp_coord[i]
        if turtleShell.get_hex_matrix_value(x, y, mode) == decimals[i]:
            nacp_coord[i] = (x, y)
        else:
            _, shell_coords = turtleShell.get_shell_coords(x, y, shells, cell_to_shells)
            found = turtleShell.find_corresponding_val(shell_coords, decimals[i], nacp_coord[i], mode)
            nacp_coord[i] = found
            
    return nacp_coord

In [21]:
def data_extract_process(nacp_coord):
    extracted_data = ""
    data_bits = ""

    for i, (x, y) in enumerate(nacp_coord):
        val = turtleShell.get_hex_matrix_value(x, y, mode="8N")
        bits = format(val & 0x7, '03b')
        data_bits += bits

        while len(data_bits) >= 8:
            byte = data_bits[:8]
            char_val = int(byte, 2)
            if char_val == 0: # Null terminator ASCII
                return extracted_data
            try:
                char = chr(char_val)
                extracted_data += char
            except:
                return extracted_data
            data_bits = data_bits[8:]

    return extracted_data

In [22]:
def encode(image_path, message_bits):
    sorted_coeffs = get_quantized_coefficients(image_path)
    nacp_coords = get_nacp(sorted_coeffs)
    print(f"NACP Coordinates: {nacp_coords}")
    print(f"NACP Length: {len(nacp_coords)}")
    print(f"Total EC: {len(nacp_coords * 3)}")

    modified_nacp_coords = data_hiding_process(message_bits, nacp_coords)
    modified_coeffs = replace_nacp(sorted_coeffs, modified_nacp_coords)
    construct_stego_file(image_path, modified_coeffs)
    print("Data embedding completed.")

In [23]:
def decode(stego_image_path):
    sorted_coeffs = get_quantized_coefficients(stego_image_path)
    nacp_coords = get_nacp(sorted_coeffs)
    extracted_bits = data_extract_process(nacp_coords)
    return extracted_bits

In [24]:
cover_folder = "cover-images/"
cover_image_path = f"{cover_folder}requantized_qf100.jpeg"
data = "The image is a grayscale closeup photograph of a mandrill, a primate species known for its distinctive facial features. The focus is on the front of the mandrills face, showcasing its symmetrical structure. The mandrill has deepsset, round eyes with a piercing gaze, surrounded by dark fur. Its nose is long and prominently structured, with vertical ridges extending down both sides. These ridges are lighter in tone compared to the surrounding facial fur. The nostrils are positioned near the lower part of the nose, just above its closed mouth, which appears neutral. The thick fur around the face creates a sense of depth and texture. Individual hair strands are visible, particularly around the cheeks and chin. The fur consists of varying shades of gray, providing contrast between different regions of the face. The background is out of focus, ensuring that the mandrills facial details remain the central visual element. The high contrast and sharpness of the image emphasize the intricate patterns on the mandrills face."
encode(cover_image_path, data)

NACP Coordinates: [(82, -22), (3, 33), (-75, 34), (78, -32), (21, 28), (27, -57), (96, -35), (-27, 52), (16, 7), (-2, 8), (-10, -17), (-17, 8), (83, -30), (36, -1), (-78, 57), (19, -27), (5, -10), (-6, 12), (-4, 27), (4, 32), (-24, 43), (-4, -3), (9, 1), (-4, 10), (-4, 3), (41, -8), (22, -8), (9, 8), (5, -14), (21, -18), (72, -17), (124, -30), (-31, 13), (-5, -71), (45, 27), (-42, 4), (-32, -13), (-71, 126), (-9, -72), (3, -4), (24, -21), (-7, -6), (24, -4), (-6, -2), (8, 16), (5, -24), (-10, 14), (16, 14), (5, 7), (-32, -25), (-18, 1), (-1, 57), (11, 10), (16, -8), (-5, -12), (1, -27), (-13, 11), (4, -14), (-18, -1), (-22, -17), (-2, 10), (107, 52), (91, 84), (43, -35), (67, -24), (-8, 25), (16, 21), (42, -48), (-28, 8), (59, -3), (-16, -14), (12, -8), (-10, 4), (39, 23), (14, 28), (31, -4), (-1, -7), (-30, 22), (30, -6), (23, -18), (36, -13), (-3, -6), (-1, -2), (13, 8), (-16, 9), (41, -5), (7, -12), (-5, -49), (11, -3), (10, 16), (-18, -9), (10, 4), (81, 44), (-24, -20), (-151, -183

In [25]:
stego_folder = "stego-images/"
stego_image_path = f"{stego_folder}stego_requantized_qf100.jpeg"
secret_data = decode(f"{stego_image_path}")
print("Extracted Data:", secret_data)

Extracted Data: The image is a grayscale closeup photograph of a mandrill, a primate species known for its distinctive facial features. The focus is on the front of the mandrills face, showcasing its symmetrical structure. The mandrill has deepsset, round eyes with a piercing gaze, surrounded by dark fur. Its nose is long and prominently structured, with vertical ridges extending down both sides. These ridges are lighter in tone compared to the surrounding facial fur. The nostrils are positioned near the lower part of the nose, just above its closed mouth, which appears neutral. The thick fur around the face creates a sense of depth and texture. Individual hair strands are visible, particularly around the cheeks and chin. The fur consists of varying shades of gray, providing contrast between different regions of the face. The background is out of focus, ensuring that the mandrills facial details remain the central visual element. The high contrast and sharpness of the image emphasize t

In [26]:
# Test performance metrics
cover_folder = "cover-images/"
stego_folder = "stego-images/"
cover_image_path = "requantized_qf100.jpeg"
stego_image_path = "stego_requantized_qf100.jpeg"
psnr_value = psnr(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
fsi_value = fsi(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
ssim_value = ssim(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
print(f"PSNR: {psnr_value} dB")
print(f"FSI: {fsi_value}")
print(f"SSIM: {ssim_value}")

Size cover: 246631
Size stego: 246583
PSNR: 65.47133990630837 dB
FSI: -0.00019462273599020398
SSIM: 0.9999956411821609
